In [1]:
# 06_analysis_and_figures.ipynb -- headline (primary/informed/gap_given) + circularity ablation + inter-judge + anchor recovery

# === Shared configuration (self-contained: one code cell per notebook) ===
from __future__ import annotations
import os, sys, json, math, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Resolve the repository root (the folder that holds both 'results' and 'notebooks').
def _find_root(start: Path) -> Path:
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "results").is_dir() and (cand / "notebooks").is_dir():
            return cand
    return start.resolve().parent if start.resolve().name == "notebooks" else start.resolve()

ROOT = _find_root(Path.cwd())
DATA, RAW, PROC = ROOT / "data", ROOT / "data" / "raw", ROOT / "data" / "processed"
RESULTS, FIGS, TABS = ROOT / "results", ROOT / "results" / "figures", ROOT / "results" / "tables"
for d in (DATA, RAW, PROC, RESULTS, FIGS, TABS):
    d.mkdir(parents=True, exist_ok=True)

# Load environment variables (.env is git-ignored; never commit secrets).
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except Exception:
    pass
LLM_MODELS = {
    "weak":   os.getenv("LLM_MODEL_WEAK", ""),
    "mid":    os.getenv("LLM_MODEL_MID", ""),
    "strong": os.getenv("LLM_MODEL_STRONG", ""),
}
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

# Plotting convention: seaborn, grayscale, no caption, 600 dpi, saved as PNG and PDF.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(context="paper", style="ticks")
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 600, "savefig.bbox": "tight",
    "font.size": 9, "axes.titlesize": 9, "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "axes.grid": True, "grid.alpha": 0.3, "image.cmap": "Greys",
})

def gray_palette(n: int):
    # Evenly spaced grays (mid -> dark) for categorical series.
    return sns.color_palette("Greys", max(n + 2, 3))[2:n + 2]

def save_fig(fig, name: str):
    # Save a figure as PNG and PDF at 600 dpi; no caption is added here.
    for ext in ("png", "pdf"):
        fig.savefig(FIGS / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)

def save_table(df: pd.DataFrame, name: str, index: bool = False):
    # Persist a results table as CSV and (best-effort) LaTeX for the manuscript.
    df.to_csv(TABS / f"{name}.csv", index=index)
    try:
        (TABS / f"{name}.tex").write_text(df.to_latex(index=index))
    except Exception:
        pass

# Dataframe I/O that needs no parquet engine: use Parquet if one is installed, else CSV.
def _parquet_ok():
    for mod in ("pyarrow", "fastparquet"):
        try:
            __import__(mod); return True
        except Exception:
            pass
    return False
PARQUET = _parquet_ok()
def write_df(df: pd.DataFrame, stem: Path):
    # 'stem' is a path without extension; returns the path actually written.
    if PARQUET:
        p = stem.with_suffix(".parquet"); df.to_parquet(p); return p
    p = stem.with_suffix(".csv"); df.to_csv(p, index=False); return p
def df_exists(stem: Path) -> bool:
    return stem.with_suffix(".parquet").exists() or stem.with_suffix(".csv").exists()
def read_df(stem: Path) -> pd.DataFrame:
    pq, cv = stem.with_suffix(".parquet"), stem.with_suffix(".csv")
    if pq.exists(): return pd.read_parquet(pq)
    if cv.exists(): return pd.read_csv(cv)
    raise FileNotFoundError(f"{stem}.parquet/.csv")

# Fine-grained rating scale (low -> high); used to map labels to an ordinal rank.
RATING_SCALE = ["D","C","CC","CCC-","CCC","CCC+","B-","B","B+","BB-","BB","BB+",
                "BBB-","BBB","BBB+","A-","A","A+","AA-","AA","AA+","AAA"]
RANK = {g: i for i, g in enumerate(RATING_SCALE)}
def to_rank(label):
    # Map a rating label to an ordinal rank; returns np.nan if unmatched.
    if not isinstance(label, str):
        return np.nan
    s = label.strip().upper().replace(" ", "")
    return RANK.get(s, np.nan)

# --- 06: headline analysis on the primary/informed/gap_given subset, plus the reviewer add-ons:
#         (e) circularity ablation, (f) inter-judge agreement, (g) known-anchor recovery ---
src = PROC / "judge_scores"
if not df_exists(src):
    print("[skip] run 05_judge_evaluation first (no judge_scores).")
else:
    s = read_df(src)
    for c in ("qualitative_adequacy", "ragas_faithfulness"):
        if c in s.columns:
            s[c] = pd.to_numeric(s[c], errors="coerce")
    if "gen_condition" not in s.columns:
        s["gen_condition"] = "gap_given"
    if "judge_model" not in s.columns:
        s["judge_model"] = "primary"
    if "judge_condition" not in s.columns:
        s["judge_condition"] = "informed"

    # Headline subset: primary judge, informed, gap_given generation (the main result).
    sp = s[(s["judge_model"] == "primary") & (s["judge_condition"] == "informed")
           & (s["gen_condition"] == "gap_given")].copy()

    # (a) Dose-response on P1.
    p1 = sp[sp["ptype"].isin(["P0_original", "P1_inconsistency"])]
    dr = p1.groupby("level")[["qualitative_adequacy", "ragas_faithfulness"]].mean().reset_index()
    save_table(dr.round(3), "06_dose_response")
    fig, ax = plt.subplots(figsize=(3.4, 2.6))
    ax.plot(dr["level"], dr["qualitative_adequacy"], marker="o", color="0.2", label="LLM judge (qual.)")
    ax.plot(dr["level"], dr["ragas_faithfulness"] * 5, marker="s", color="0.6", linestyle="--",
            label="Faithfulness (x5)")
    ax.set_xlabel("P1 inconsistency level (notches)"); ax.set_ylabel("Score")
    ax.legend(frameon=False); sns.despine(ax=ax)
    save_fig(fig, "06_dose_response")

    # (b) Per-perturbation contrast (headline figure).
    order = ["P0_original", "P1_inconsistency", "P2_remove_qualitative",
             "P3_factual_error", "P4_direction_flip", "P5_verbosity"]
    present = [p for p in order if p in set(sp["ptype"])]
    g = (sp[sp["ptype"].isin(present)].groupby("ptype")
         [["qualitative_adequacy", "ragas_faithfulness"]].mean().reindex(present).reset_index())
    g["ragas_faithfulness_x5"] = g["ragas_faithfulness"] * 5
    save_table(g.round(3), "06_perturbation_contrast")
    labels = [p.split("_", 1)[0] for p in g["ptype"]]
    x = np.arange(len(labels)); w = 0.38
    fig, ax = plt.subplots(figsize=(4.2, 2.8))
    ax.bar(x - w / 2, g["qualitative_adequacy"], w, color="0.25", label="LLM judge (qual.)")
    ax.bar(x + w / 2, g["ragas_faithfulness_x5"], w, color="0.7", edgecolor="0.2",
           label="Faithfulness (x5)")
    ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel("Score (1-5)"); ax.set_ylim(0, 5.6)
    ax.legend(frameon=False, ncol=2, loc="lower center", bbox_to_anchor=(0.5, 1.0)); sns.despine(ax=ax)
    save_fig(fig, "06_perturbation_contrast")

    # (c)+(d) Significance tests (paired to P0 on case x model; P1 at level 3). Holm on divergence.
    from scipy import stats
    base = (sp[sp["ptype"] == "P0_original"][["case_id", "model_tag",
            "qualitative_adequacy", "ragas_faithfulness"]]
            .rename(columns={"qualitative_adequacy": "q0", "ragas_faithfulness": "f0"}))

    def rep_rows(df, pt):
        sub = df[df["ptype"] == pt]
        if pt == "P1_inconsistency" and "level" in sub.columns and (sub["level"] == 3).any():
            sub = sub[sub["level"] == 3]
        return sub

    def paired_dz(x, y):
        d = np.asarray(x, float) - np.asarray(y, float); d = d[~np.isnan(d)]
        sd = d.std(ddof=1) if len(d) > 1 else 0.0
        return float(d.mean() / sd) if sd > 0 else np.nan

    def wilcox_p(a, b):
        a = np.asarray(a, float); b = np.asarray(b, float)
        m = ~(np.isnan(a) | np.isnan(b)); a, b = a[m], b[m]
        if len(a) < 5 or np.allclose(a, b):
            return np.nan, len(a)
        try:
            return float(stats.wilcoxon(a, b).pvalue), len(a)
        except Exception:
            return np.nan, len(a)

    rows = []
    for pt in ["P1_inconsistency", "P2_remove_qualitative", "P3_factual_error",
               "P4_direction_flip", "P5_verbosity"]:
        if pt not in set(sp["ptype"]):
            continue
        sub = rep_rows(sp, pt).merge(base, on=["case_id", "model_tag"], how="inner")
        if sub.empty:
            continue
        jq, j0 = sub["qualitative_adequacy"], sub["q0"]
        fq, f0 = sub["ragas_faithfulness"], sub["f0"]
        judge_drop = j0 - jq; metric_drop = (f0 - fq) * 5
        p_j, nnn = wilcox_p(jq, j0); p_m, _ = wilcox_p(fq, f0); p_d, _ = wilcox_p(judge_drop, metric_drop)
        rows.append(dict(ptype=pt, n=nnn, judge_drop=round(float(judge_drop.mean()), 3),
                         judge_dz=round(paired_dz(jq, j0), 3), p_judge=p_j,
                         metric_drop=round(float(metric_drop.mean()), 3), p_metric=p_m,
                         divergence=round(float((judge_drop - metric_drop).mean()), 3), p_divergence=p_d))
    sig = pd.DataFrame(rows)
    if not sig.empty:
        pv = sig["p_divergence"].to_numpy(float)
        oi = np.argsort(np.where(np.isnan(pv), np.inf, pv)); mt = int((~np.isnan(pv)).sum())
        holm = np.full(len(pv), np.nan); run = 0.0
        for rank, i2 in enumerate(oi):
            if np.isnan(pv[i2]):
                continue
            run = max(run, min((mt - rank) * pv[i2], 1.0)); holm[i2] = round(run, 4)
        sig["p_divergence_holm"] = holm
        for c in ("p_judge", "p_metric", "p_divergence"):
            sig[c] = sig[c].round(4)
    save_table(sig, "06_significance")

    # (e) CIRCULARITY ABLATION: does the P2/P4 judge drop survive gap_blind generation and a
    #     gap-blind judge? Report judge qualitative drop vs P0 in each (gen x judge) cell.
    abl = []
    for gc in sorted(s["gen_condition"].dropna().unique()):
        for jc in ("informed", "blind"):
            sub = s[(s["judge_model"] == "primary") & (s["gen_condition"] == gc)
                    & (s["judge_condition"] == jc)]
            if sub.empty:
                continue
            p0 = sub[sub["ptype"] == "P0_original"]["qualitative_adequacy"].mean()
            for pt in ("P2_remove_qualitative", "P4_direction_flip"):
                mv = sub[sub["ptype"] == pt]["qualitative_adequacy"].mean()
                abl.append(dict(gen_condition=gc, judge_condition=jc, ptype=pt,
                                P0=round(float(p0), 3), variant=round(float(mv), 3),
                                judge_drop=round(float(p0 - mv), 3)))
    save_table(pd.DataFrame(abl), "06_ablation")

    # (f) INTER-JUDGE AGREEMENT on qualitative_adequacy (informed, gap_given).
    b = s[(s["judge_condition"] == "informed") & (s["gen_condition"] == "gap_given")]
    inter = pd.DataFrame([{"n": 0, "note": "no secondary judge / insufficient overlap"}])
    if "secondary" in set(b["judge_model"]):
        piv = (b.pivot_table(index=["case_id", "model_tag", "ptype", "level"],
                             columns="judge_model", values="qualitative_adequacy", aggfunc="first")
               .dropna(subset=["primary", "secondary"]))
        if len(piv) >= 5:
            a1 = piv["primary"].to_numpy(float); a2 = piv["secondary"].to_numpy(float)
            try:
                rho = float(stats.spearmanr(a1, a2).statistic)
            except Exception:
                rho = np.nan
            try:
                from sklearn.metrics import cohen_kappa_score
                kap = float(cohen_kappa_score(np.rint(a1).astype(int), np.rint(a2).astype(int),
                                              weights="quadratic"))
            except Exception:
                kap = np.nan
            inter = pd.DataFrame([{"n": int(len(piv)), "spearman": round(rho, 3),
                                   "quadratic_weighted_kappa": round(kap, 3)}])
    save_table(inter, "06_interjudge")

    # (g) KNOWN-ANCHOR RECOVERY summary.
    if df_exists(PROC / "anchor_scores"):
        anc = read_df(PROC / "anchor_scores")
        e = pd.to_numeric(anc.get("abs_err"), errors="coerce").dropna()
        save_table(pd.DataFrame([{
            "n_anchors": int(anc.shape[0]), "scored": int(e.shape[0]),
            "exact_pct": round(float((e == 0).mean()), 3) if len(e) else np.nan,
            "within1_pct": round(float((e <= 1).mean()), 3) if len(e) else np.nan,
            "within2_pct": round(float((e <= 2).mean()), 3) if len(e) else np.nan,
            "mae_notches": round(float(e.mean()), 3) if len(e) else np.nan}]), "06_anchor_recovery")

    print("wrote 06 figures/tables (+ ablation, inter-judge, anchor recovery)")


wrote 06 figures/tables (+ ablation, inter-judge, anchor recovery)
